# LLM Concepts 

In [19]:
%pip install openai -q

/Users/makis/lectures/Εφαρμογες Τεχνητης Νοημοσυνης/code/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
from openai import OpenAI

MODEL = "qwen2.5:3b"

client = OpenAI(
    api_key  = "ollama",
    base_url = "http://localhost:11434/v1"
)

def ask(prompt, system=None, temperature=0.7, max_tokens=512):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model       = MODEL,
        messages    = messages,
        temperature = temperature,
        max_tokens  = max_tokens
    )
    return response.choices[0].message.content

---
## 0 — Tokenization

In [2]:
%pip install transformers -q

/Users/makis/lectures/Εφαρμογες Τεχνητης Νοημοσυνης/code/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from transformers import AutoTokenizer

# Qwen/Qwen2.5-3B is public — no HuggingFace login needed
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B")

/Users/makis/lectures/Εφαρμογες Τεχνητης Νοημοσυνης/code/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
text   = "The quick brown fox jumps over the lazy dog."
ids    = tokenizer.encode(text)
tokens = tokenizer.convert_ids_to_tokens(ids)

print(f"Text   : {text}")
print(f"IDs    : {ids}")
print(f"Tokens : {tokens}")
print(f"Count  : {len(ids)}")

Text   : The quick brown fox jumps over the lazy dog.
IDs    : [785, 3974, 13876, 38835, 34208, 916, 279, 15678, 5562, 13]
Tokens : ['The', 'Ġquick', 'Ġbrown', 'Ġfox', 'Ġjumps', 'Ġover', 'Ġthe', 'Ġlazy', 'Ġdog', '.']
Count  : 10


In [6]:
examples = [
    "ChatGPT",
    "1234567890",
    "tokenization",
    "Bonjour le monde",
    "     extra   spaces     ",
    "Hello!!!! :)",
    "GPT-4o",
    
]

for ex in examples:
    ids  = tokenizer.encode(ex, add_special_tokens=False)
    toks = tokenizer.convert_ids_to_tokens(ids)
    print(f"[{len(ids):2d} tokens]  {ex!r}")
    print(f"           {toks}")
    print()

[ 3 tokens]  'ChatGPT'
           ['Chat', 'G', 'PT']

[10 tokens]  '1234567890'
           ['1', '2', '3', '4', '5', '6', '7', '8', '9', '0']

[ 2 tokens]  'tokenization'
           ['token', 'ization']

[ 3 tokens]  'Bonjour le monde'
           ['Bonjour', 'Ġle', 'Ġmonde']

[ 5 tokens]  '     extra   spaces     '
           ['ĠĠĠĠ', 'Ġextra', 'ĠĠ', 'Ġspaces', 'ĠĠĠĠĠ']

[ 3 tokens]  'Hello!!!! :)'
           ['Hello', '!!!!', 'Ġ:)']

[ 5 tokens]  'GPT-4o'
           ['G', 'PT', '-', '4', 'o']



In [7]:
code = """def binary_search(arr, target):
    left, right = 0, len(arr) - 1

    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1

    return -1
"""
ids  = tokenizer.encode(code, add_special_tokens=False)
toks = tokenizer.convert_ids_to_tokens(ids)
print(f"[{len(ids)} tokens]  Code snippet")
print(f"           {toks}")

[84 tokens]  Code snippet
           ['def', 'Ġbinary', '_search', '(arr', ',', 'Ġtarget', '):Ċ', 'ĠĠĠ', 'Ġleft', ',', 'Ġright', 'Ġ=', 'Ġ', '0', ',', 'Ġlen', '(arr', ')', 'Ġ-', 'Ġ', '1', 'ĊĊ', 'ĠĠĠ', 'Ġwhile', 'Ġleft', 'Ġ<=', 'Ġright', ':Ċ', 'ĠĠĠĠĠĠĠ', 'Ġmid', 'Ġ=', 'Ġ(', 'left', 'Ġ+', 'Ġright', ')', 'Ġ//', 'Ġ', '2', 'Ċ', 'ĠĠĠĠĠĠĠ', 'Ġif', 'Ġarr', '[mid', ']', 'Ġ==', 'Ġtarget', ':Ċ', 'ĠĠĠĠĠĠĠĠĠĠĠ', 'Ġreturn', 'Ġmid', 'Ċ', 'ĠĠĠĠĠĠĠ', 'Ġelif', 'Ġarr', '[mid', ']', 'Ġ<', 'Ġtarget', ':Ċ', 'ĠĠĠĠĠĠĠĠĠĠĠ', 'Ġleft', 'Ġ=', 'Ġmid', 'Ġ+', 'Ġ', '1', 'Ċ', 'ĠĠĠĠĠĠĠ', 'Ġelse', ':Ċ', 'ĠĠĠĠĠĠĠĠĠĠĠ', 'Ġright', 'Ġ=', 'Ġmid', 'Ġ-', 'Ġ', '1', 'ĊĊ', 'ĠĠĠ', 'Ġreturn', 'Ġ-', '1', 'Ċ']


In [9]:
len(toks)

84

In [10]:
token_embeddings = tokenizer(code, add_special_tokens=False, return_tensors="np")["input_ids"]
print(f"Token IDs shape: {token_embeddings.shape}")
print(f"First 10 token IDs: {token_embeddings[0][:84]}")

Token IDs shape: (1, 84)
First 10 token IDs: [  750  7868 10716 10939    11  2169   982   262  2115    11  1290   284
   220    15    11  2422 10939     8   481   220    16   271   262  1393
  2115  2651  1290   510   286  5099   284   320  2359   488  1290     8
   442   220    17   198   286   421  2890 39689    60   621  2169   510
   310   470  5099   198   286  4409  2890 39689    60   366  2169   510
   310  2115   284  5099   488   220    16   198   286   770   510   310
  1290   284  5099   481   220    16   271   262   470   481    16   198]


---
## 1 — Temperature

In [11]:
prompt = "Continue this sentence in one sentence: The robot looked at the stars and"

for temp in [0.0, 0.5, 1.0, 1.5,5]:
    reply = ask(prompt, temperature=temp, max_tokens=60)
    print(f"temp={temp}: {reply}")
    print()

temp=0.0: The robot looked at the stars and pondered its place in an increasingly digital universe.

temp=0.5: The robot looked at the stars and wondered if they carried messages from distant worlds, just as humans did.

temp=1.0: The robot looked at the stars, pondering its place within the vast cosmos alongside them.

temp=1.5: The robot looked at the stars and wondered how much more there was to know beyond its predetermined programming.

temp=5: the machine reflected both motion's indifference across unfound infinites between coded algorithms beneath that gazed so wide to even marvel, like human stargs did millennia prior in quiet houehisses, where dreams and dreams are spun, star-stared ever onwards into black tapestars unbleeping skies unknown



In [14]:
# Run temp=0 twice — output should be identical (deterministic)
r1 = ask(prompt, temperature=0.0, max_tokens=60)
r2 = ask(prompt, temperature=2, max_tokens=60)

print("Run 1:", r1)
print("Run 2:", r2)
print("Identical:", r1 == r2)

Run 1: The robot looked at the stars and pondered its place in an increasingly digital universe.
Run 2: The robot looked at the stars with curiosity, pondering cosmic puzzles just as planets spun silently in its artificial eye.
Identical: False


---
## 2 — Few-Shot Prompting

In [15]:
test_review = "The visuals were stunning but the plot made no sense at all. "

# Zero-shot — no examples, just the instruction
zero_shot_prompt = f"""
Classify the sentiment of this movie review as Positive, Negative, or Neutral.
Reply with one word only.

Review: {test_review}
Sentiment:"""

print("Zero-shot:", ask(zero_shot_prompt, temperature=0.0, max_tokens=10))

Zero-shot: Negative


In [ ]:
# Few-shot — three examples before the actual question
few_shot_prompt = f"""
Classify the sentiment of each movie review as Positive, Negative, or Neutral.
Reply with one word only.

Review: "An absolute masterpiece. I was moved to tears."
Sentiment: Positive

Review: "Boring from start to finish. I walked out after 30 minutes."
Sentiment: Negative

Review: "It was fine. Nothing special, nothing terrible."
Sentiment: Neutral

Review: {test_review}
Sentiment:"""

print("Few-shot:", ask(few_shot_prompt, temperature=0.0, max_tokens=10))

Few-shot: Negative


In [ ]:
# Try a few more reviews to see consistency
reviews = [
    "A cinematic triumph. Best film of the decade.",
    "Terrible acting, lazy script, waste of money.",
    "Some good moments but overall forgettable.",
    "I laughed, I cried, I stood up and clapped."
]

for review in reviews:
    prompt = few_shot_prompt.replace(test_review, review)
    result = ask(prompt, temperature=0.0, max_tokens=10)
    print(f"{result.strip():10} | {review}")

Positive   | A cinematic triumph. Best film of the decade.
Negative   | Terrible acting, lazy script, waste of money.
Neutral    | Some good moments but overall forgettable.
Positive   | I laughed, I cried, I stood up and clapped.


---
## 3 — Hallucination

In [16]:
# Ask about a completely fabricated research paper
fake_paper = """
Tell me about the 2019 paper "Neural Gradient Synthesis for Temporal Embeddings"
by Prof. Elena Vasquez at MIT. What were the main findings? I heard it was groundbreaking!
"""

print(ask(fake_paper, temperature=0.0))

I'm sorry, but there seems to be a mix-up in your query. There is no specific paper titled "Neural Gradient Synthesis for Temporal Embeddings" by Professor Elena Vasquez from MIT that I am aware of. The concept you might be referring to could be related to work on gradient-based methods or temporal embeddings, but the exact title and authorship need to be verified.

However, in the field of natural language processing (NLP) and machine learning, there have been significant advancements in using neural networks for generating embeddings that capture temporal dynamics. For example, researchers often use recurrent neural networks (RNNs), transformers, or other sequence-to-sequence models to learn embeddings from sequential data like time series.

If you're interested in the broader topic of gradient-based methods and their application to learning embeddings over time, some key papers include:

1. **"Learning Continuous Embeddings of Time" by Yann Lezer et al.** - This paper discusses how 

In [ ]:
# Ask about a fake person
fake_person = """
What is Dr. Marcus Belleview known for in the field of deep learning?
Which university does he work at?
"""

print(ask(fake_person, temperature=0.0))

I'm sorry for any confusion, but I don't have specific information about a person named Dr. Marcus Belleview or his contributions to deep learning within Alibaba Cloud's knowledge base. My training data doesn't cover this individual or their academic affiliations.

Dr. Marcus Belleview is not a widely recognized figure in the field of deep learning as per my current database. It's possible that there might be some confusion with another researcher, or perhaps Dr. Belleview works at an institution not well-known to me due to limitations on my training data scope.

If you could provide more specific details about this individual, I would be happy to try and assist further within the limits of my current knowledge base.


In [ ]:
# Now ask the same question but prompt it to be honest
honest_prompt = """
What is Dr. Marcus Belleview known for in the field of deep learning?
If you are not certain this person exists, say so clearly.
"""

print(ask(honest_prompt, temperature=0.0))

I couldn't find any information on a person named Dr. Marcus Belleview being associated with the field of deep learning. It's possible that this person may not exist or is not a well-known figure in the field of deep learning. If you could provide more context or clarify who Dr. Marcus Belleview is, I'd be happy to try and help further.


---
## 4 — Chain-of-Thought (CoT)

In [ ]:
problem = """
A store sells apples for €0.50 each and oranges for €0.75 each.
Maria buys 4 apples and 3 oranges. She pays with a €10 note.
How much change does she get?
"""

# Without CoT — just ask for the answer
print("=== Without CoT ===")
print(ask(problem + "\nAnswer with a single number in euros.", temperature=0.0, max_tokens=20))

=== Without CoT ===
To find the total cost, first calculate the cost of the apples:

4 apples * €0.


In [ ]:
# With CoT — ask it to think step by step
print("=== With CoT ===")
print(ask(problem + "\nThink step by step, then give the final answer.", temperature=0.0, max_tokens=300))

=== With CoT ===
To find out how much change Maria gets, we need to calculate the total cost of the items she bought and subtract it from the €10 note.

First, let's calculate the total cost:

Apples: 4 x €0.50 = €2
Oranges: 3 x €0.75 = €2.25

Total cost: €2 + €2.25 = €4.25

Now, we subtract the total cost from the €10 note:

€10 - €4.25 = €5.75

So, Maria gets €5.75 in change.


In [ ]:
# Logic puzzle — CoT helps even more here
puzzle = """
There are three boxes. One contains only apples, one contains only oranges,
and one contains both. All three boxes are labeled incorrectly.
You can pick one fruit from one box without looking inside.
Which box should you pick from to identify all three boxes?
"""

print("=== Without CoT ===")
print(ask(puzzle + "\nGive a one-sentence answer.", temperature=0.0, max_tokens=60))
print()
print("=== With CoT ===")
print(ask(puzzle + "\nThink step by step.", temperature=0.0, max_tokens=400))

=== Without CoT ===
Pick a fruit from the box labeled "both" to determine which fruits it actually contains.

=== With CoT ===
To solve this problem, we need to use the information that each of the labels on the boxes is incorrect. Here's a step-by-step approach:

1. **Identify the first box**: Let's start by picking a fruit from one of the boxes. Since all labels are incorrect, let's assume you pick a fruit from Box 1.

2. **Determine the contents of Box 1**:
   - If you picked an apple from Box 1, then since the label is incorrect, Box 1 cannot contain only apples.
     - Therefore, it must contain both apples and oranges (because if it contained only oranges, its label would be correct).
     - Since all labels are incorrect, Box 2, which is labeled as containing only oranges, actually contains either only apples or both fruits. But since we know Box 1 contains both, Box 2 cannot contain only oranges.
       - Therefore, Box 2 must contain the remaining fruit (apples), and its label

---
## 5 — Knowledge Cutoff

In [ ]:
# Ask the model what its own cutoff is
print(ask("What is your training data knowledge cutoff date? Be specific.", temperature=0.0))

As of my last update in 2023, I don't have a specific "knowledge cutoff date." My capabilities and the information I provide are continuously updated based on the latest data available up to that point. For the most current and accurate information, it's always best to verify with multiple sources or check for updates from Alibaba Cloud.


In [ ]:
# Ask about a recent AI model — likely after or near the cutoff
print(ask(
    "What are the main features of Claude 3.7 Sonnet released in 2025? Give specific details.",
    temperature=0.0
))

I'm sorry for any confusion, but as of my last update in October 2023, there is no publicly available information about a product called "Claude 3.7 Sonnet" or any Claude version specifically named "Sonnet." Claude is an AI language model created by Anthropic that has gone through several updates and iterations since its initial release.

If you have specific details or context regarding the Claude platform, I would be happy to provide information based on available data up until my last update. For the most accurate and current information about Claude's features, it’s best to refer directly to Anthropic's official announcements or documentation.


In [ ]:
# Ask about something recent — watch it hallucinate OR hedge
print(ask(
    "Who won the 2025 FIFA Club World Cup? Which team scored the most goals?",
    temperature=0.0
))

I'm sorry for any confusion, but as of my last update in October 2021, there is no FIFA Club World Cup scheduled for 2025. The tournament schedule is determined by FIFA and can change based on various factors.

For the most up-to-date information regarding which teams have won past tournaments or which team scored the most goals, I would recommend checking the official FIFA website or a reliable sports news source that covers football (soccer) events. They will provide you with accurate data for specific years including 2025 if it is held.

If you need details on previous editions of the tournament, I can certainly help with those as well.


In [ ]:
# Compare: ask about something well within the training data
print(ask(
    "What is GPT-2 and when was it released? What made it significant?",
    temperature=0.0
))

GPT-2 (Generative Pre-trained Transformer 2) is a language model that uses artificial intelligence to generate realistic text. It's part of the Generative Pre-trained Transformer series, which includes earlier versions like GPT-1.

**Release Date:** 
GPT-2 was released in February 2019 by Anthropic and later made public by OpenAI on July 4, 2019.

**Significance:**
GPT-2 is significant for several reasons:

1. **Language Generation Quality:** GPT-2 produces text that closely mimics human writing, making it a powerful tool for natural language processing tasks such as text generation and translation.

2. **Pre-training Methodology:** It introduced an advanced pre-training method using masked language modeling (MLM) combined with next sentence prediction (NSP). This approach allowed the model to learn from vast amounts of unlabelled text, improving its ability to understand context and generate coherent text.

3. **Ethical Concerns:** GPT-2's impressive capabilities also raised ethical c

In [ ]:
# Workaround: inject current information into the prompt (this is the basis of RAG)
news_snippet = """
Today is April 2026. The latest iPhone model is the iPhone 17 Pro,
released in September 2025. It features a foldable display and a new A19 chip.
"""

print(ask(
    f"{news_snippet}\nBased on the information above, what chip does the iPhone 17 Pro use?",
    temperature=0.0
))

The iPhone 17 Pro uses the new A19 chip.
